# Table III — kill-chain coverage of zero-shot benchmarks and CAPTure

## Standalone reproduction

This notebook reproduces the main-paper **Table III** with the same dataset order, tactic columns, and values as the Overleaf manuscript. It is fully standalone and does **not** require TGCM model code, checkpoints, GPU execution, CAPTure raw CSV files, or another notebook.

### Environment

- Python 3.12
- pandas 2.2
- JupyterLab / IPython
- CPU only; CUDA is not required.

For this notebook alone, `pip install pandas==2.2.*` is sufficient.

### Required data

1. `kill_chain_mapping.json` — disclosed zero-shot technique-to-ATT&CK-tactic mapping for ATLAS, NODLINK, ProvCon, DARPA TC-E3, and DARPA TC-E5.
2. `capture_kill_chain_coverage.json` — CAPTure profile-level kill-chain coverage summarized from the appendix mapping.

Both files are downloaded automatically from a pinned TGCM_Website commit if they are not already next to the notebook.

### Table contract

The output columns are exactly: `Dataset`, `Init. Access`, `Execution`, `Persistence`, `Def. Evasion`, `Cred. Access`, `Discovery`, `Lat. Move.`, `Collection`, `C2`, `Exfiltration`. Counts are the number of **unique techniques** mapped to each tactic. The final cell asserts exact equality with the Table III values in the Overleaf manuscript.


In [ ]:
from pathlib import Path
import json
import urllib.request
import pandas as pd

SUPPORT_REV = "ece4c0d85cd19ab50068e1f3e6696005c0cb5311"
BASE_URL = f"https://raw.githubusercontent.com/Irish-kw/TGCM_Website/{SUPPORT_REV}/reproduction/paper_metadata"

FILES = {
    "zero_shot": (Path.cwd() / "kill_chain_mapping.json", f"{BASE_URL}/kill_chain_mapping.json"),
    "capture": (Path.cwd() / "capture_kill_chain_coverage.json", f"{BASE_URL}/capture_kill_chain_coverage.json"),
}

for label, (path, url) in FILES.items():
    if not path.is_file():
        print(f"Downloading {label} metadata...")
        urllib.request.urlretrieve(url, path)

zero_shot_payload = json.loads(FILES["zero_shot"][0].read_text(encoding="utf-8"))
capture_payload = json.loads(FILES["capture"][0].read_text(encoding="utf-8"))
print(f"Loaded {len(zero_shot_payload)} zero-shot datasets and {capture_payload['dataset']} metadata.")


In [ ]:
TACTICS = [
    "Initial Access",
    "Execution",
    "Persistence",
    "Defense Evasion",
    "Credential Access",
    "Discovery",
    "Lateral Movement",
    "Collection",
    "C2",
    "Exfiltration",
]

DISPLAY_NAMES = {
    "Initial Access": "Init. Access",
    "Execution": "Execution",
    "Persistence": "Persistence",
    "Defense Evasion": "Def. Evasion",
    "Credential Access": "Cred. Access",
    "Discovery": "Discovery",
    "Lateral Movement": "Lat. Move.",
    "Collection": "Collection",
    "C2": "C2",
    "Exfiltration": "Exfiltration",
}

rows = []
detail_rows = []
for dataset, phases in zero_shot_payload.items():
    row = {"Dataset": dataset}
    for tactic in TACTICS:
        techniques = sorted(set(phases.get(tactic, [])))
        row[DISPLAY_NAMES[tactic]] = len(techniques)
        for technique in techniques:
            detail_rows.append({"Dataset": dataset, "Tactic": tactic, "Technique": technique})
    rows.append(row)

capture_row = {"Dataset": capture_payload["dataset"]}
for tactic in TACTICS:
    capture_row[DISPLAY_NAMES[tactic]] = int(capture_payload["coverage"].get(tactic, 0))
rows.append(capture_row)

coverage = pd.DataFrame(rows, columns=["Dataset", *DISPLAY_NAMES.values()])
zero_shot_mapping_detail = pd.DataFrame(detail_rows)
coverage


In [ ]:
EXPECTED_OVERLEAF_TABLE_III = pd.DataFrame([
    ["ATLAS",       0, 3, 1, 3, 0, 1, 0, 0, 2, 0],
    ["NODLINK",     0, 1, 0, 0, 0, 9, 0, 1, 1, 0],
    ["ProvCon",     1, 0, 1, 0, 1, 1, 0, 0, 1, 0],
    ["DARPA TC-E3", 2, 6, 3, 1, 2, 9, 0, 1, 3, 1],
    ["DARPA TC-E5", 1, 4, 1, 3, 2, 9, 0, 2, 4, 2],
    ["CAPTure",     8, 28, 24, 14, 0, 58, 11, 17, 13, 1],
], columns=coverage.columns)

pd.testing.assert_frame_equal(
    coverage.reset_index(drop=True),
    EXPECTED_OVERLEAF_TABLE_III,
    check_dtype=False,
)
print("PASS: reproduced values exactly match Overleaf Table III.")
coverage.to_csv("table03_kill_chain_coverage.csv", index=False)
coverage


In [ ]:
# Complete zero-shot technique-to-tactic detail used to derive the first five rows.
zero_shot_mapping_detail
